<!-- journey-badges -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NolaNexus/LMp2LMa/blob/main/track-02-dl-programmer/08_gnn.ipynb)
[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/NolaNexus/LMp2LMa/blob/main/track-02-dl-programmer/08_gnn.ipynb)
[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/NolaNexus/LMp2LMa)

# Graph Neural Networks: GCN, GAT, GraphSAGE

**Module 4 - Notebook 8** | Skillsoft Track 2: DL Programmer

## Objectives
- GCN/GAT/GraphSAGE on citation networks
- Node classification
- Link prediction
- Graph classification

In [ ]:
# Setup
!pip install torch-geometric==2.4.0 -q 2>/dev/null || pip install torch-geometric -q

import torch
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GATConv, GCNConv, SAGEConv
from torch_geometric.transforms import NormalizeFeatures

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 1. Load Citation Dataset (Cora)

In [ ]:
# Cora: 2708 papers, 5429 citations, 7 classes, 1433 features
dataset = Planetoid(root="./data/Planetoid", name="Cora", transform=NormalizeFeatures())
data = dataset[0].to(device)

print(f"Dataset: {dataset}")
print(f"Nodes: {data.num_nodes}")
print(f"Edges: {data.num_edges}")
print(f"Features: {data.num_node_features}")
print(f"Classes: {dataset.num_classes}")
print(f"Train mask: {data.train_mask.sum().item()}")
print(f"Val mask: {data.val_mask.sum().item()}")
print(f"Test mask: {data.test_mask.sum().item()}")

## 2. GCN (Graph Convolutional Network)

In [ ]:
class GCN(torch.nn.Module):
    def __init__(self, num_features, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, num_classes)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, training=self.training)
        x = self.conv2(x, edge_index)
        return x


gcn = GCN(dataset.num_node_features, 16, dataset.num_classes).to(device)
optimizer = torch.optim.Adam(gcn.parameters(), lr=0.01, weight_decay=5e-4)


def train_gcn():
    gcn.train()
    optimizer.zero_grad()
    out = gcn(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()


def test_gcn():
    gcn.eval()
    out = gcn(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    accs = []
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        acc = (pred[mask] == data.y[mask]).sum().item() / mask.sum().item()
        accs.append(acc)
    return accs


for epoch in range(200):
    loss = train_gcn()
    if epoch % 50 == 0:
        train_acc, val_acc, test_acc = test_gcn()
        print(
            f"Epoch {epoch}: Loss={loss:.4f}, Train={train_acc:.4f}, Val={val_acc:.4f}, Test={test_acc:.4f}"
        )

## 3. GAT (Graph Attention Network)

In [ ]:
class GAT(torch.nn.Module):
    def __init__(self, num_features, hidden_dim, num_classes, heads=8):
        super().__init__()
        self.conv1 = GATConv(num_features, hidden_dim, heads=heads, dropout=0.6)
        self.conv2 = GATConv(hidden_dim * heads, num_classes, heads=1, concat=False, dropout=0.6)

    def forward(self, x, edge_index):
        x = F.dropout(x, p=0.6, training=self.training)
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=0.6, training=self.training)
        x = self.conv2(x, edge_index)
        return x


gat = GAT(dataset.num_node_features, 8, dataset.num_classes).to(device)
optimizer = torch.optim.Adam(gat.parameters(), lr=0.005, weight_decay=5e-4)


def train_gat():
    gat.train()
    optimizer.zero_grad()
    out = gat(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()


def test_gat():
    gat.eval()
    out = gat(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    accs = []
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        acc = (pred[mask] == data.y[mask]).sum().item() / mask.sum().item()
        accs.append(acc)
    return accs


for epoch in range(200):
    loss = train_gat()
    if epoch % 50 == 0:
        train_acc, val_acc, test_acc = test_gat()
        print(
            f"Epoch {epoch}: Loss={loss:.4f}, Train={train_acc:.4f}, Val={val_acc:.4f}, Test={test_acc:.4f}"
        )

## 4. GraphSAGE

In [ ]:
class GraphSAGE(torch.nn.Module):
    def __init__(self, num_features, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = SAGEConv(num_features, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, num_classes)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, training=self.training)
        x = self.conv2(x, edge_index)
        return x


sage = GraphSAGE(dataset.num_node_features, 16, dataset.num_classes).to(device)
optimizer = torch.optim.Adam(sage.parameters(), lr=0.01, weight_decay=5e-4)


def train_sage():
    sage.train()
    optimizer.zero_grad()
    out = sage(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()


def test_sage():
    sage.eval()
    out = sage(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    accs = []
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        acc = (pred[mask] == data.y[mask]).sum().item() / mask.sum().item()
        accs.append(acc)
    return accs


for epoch in range(200):
    loss = train_sage()
    if epoch % 50 == 0:
        train_acc, val_acc, test_acc = test_sage()
        print(
            f"Epoch {epoch}: Loss={loss:.4f}, Train={train_acc:.4f}, Val={val_acc:.4f}, Test={test_acc:.4f}"
        )

## 5. Comparison & Link Prediction

In [ ]:
# Compare all models
models = {"GCN": gcn, "GAT": gat, "GraphSAGE": sage}

for name, model in models.items():
    train_acc, val_acc, test_acc = (
        test_gcn() if name == "GCN" else test_gat() if name == "GAT" else test_sage()
    )
    print(f"{name}: Train={train_acc:.4f}, Val={val_acc:.4f}, Test={test_acc:.4f}")

In [ ]:
# Link Prediction with GCN
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.utils import negative_sampling

# Split edges
transform = RandomLinkSplit(
    num_val=0.05, num_test=0.1, is_undirected=True, add_negative_train_samples=False
)
train_data, val_data, test_data = transform(data)


class LinkPredictor(torch.nn.Module):
    def __init__(self, num_features, hidden_dim):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x

    def decode(self, z, edge_label_index):
        src, dst = edge_label_index
        return (z[src] * z[dst]).sum(dim=-1)


lp_model = LinkPredictor(dataset.num_node_features, 64).to(device)
optimizer = torch.optim.Adam(lp_model.parameters(), lr=0.01)
criterion = torch.nn.BCEWithLogitsLoss()


def train_lp():
    lp_model.train()
    optimizer.zero_grad()
    z = lp_model(train_data.x, train_data.edge_index)

    # Positive edges
    pos_out = lp_model.decode(z, train_data.edge_label_index)
    pos_loss = criterion(pos_out, train_data.edge_label.float())

    # Negative edges
    neg_edge_index = negative_sampling(
        train_data.edge_index,
        num_nodes=train_data.num_nodes,
        num_neg_samples=train_data.edge_label_index.size(1),
    )
    neg_out = lp_model.decode(z, neg_edge_index)
    neg_loss = criterion(neg_out, torch.zeros_like(neg_out))

    loss = pos_loss + neg_loss
    loss.backward()
    optimizer.step()
    return loss.item()


for epoch in range(50):
    loss = train_lp()
    if epoch % 10 == 0:
        print(f"Epoch {epoch}: Link Prediction Loss = {loss:.4f}")